# Current upper PPO best: QoS and demand before/after

Evaluates the current `upper_ppo_best.zip` checkpoint from `models/upper_ppo_mixed_only_fixed_reward_10k/run_20260706_212506` and produces CSV tables plus plots for:

- demand PRBs before/after by gNB and slice
- used PRBs before/after by gNB and slice
- network QoS and handovers
- per-slice QoS
- per-gNB/per-slice QoS heatmaps
- learned bias/action summaries

Note: demand and used PRBs have true `start`/`end` fields inside each upper window. QoS is logged for the post-action measurement window, so the QoS before/after comparison uses `Zero action` as the no-policy baseline and `Trained PPO` as the after-policy controller on the same scenario.

In [ ]:
import os
from pathlib import Path

if not Path('train_upper_ppo_3gnb.py').exists():
    os.chdir(Path.cwd().parent)

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from stable_baselines3 import PPO

from run_zero_action_baseline import ZeroActionPolicy, build_args
from train_upper_ppo_3gnb import evaluate_upper_policy, make_env

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)

RUN_DIR = Path('models/upper_ppo_mixed_only_fixed_reward_10k/run_20260706_212506')
MODEL_PATH = RUN_DIR / 'upper_ppo_best.zip'
CONFIG_PATH = RUN_DIR / 'config.json'
OUT_DIR = Path('results/current_upper_ppo_best_qos_demand_before_after')

# Default: inspect the scenario this model was trained on. Add more names or set to ALL_CONTROLLABLE if desired.
ALL_CONTROLLABLE = [
    'jain_balance_controllable',
    'jain_control_urllc',
    'jain_control_mmtc',
    'jain_control_embb_urllc',
    'jain_control_embb_mmtc',
    'jain_control_urllc_mmtc',
    'jain_control_outer_congested',
    'jain_control_mixed',
]
SCENARIOS = ['jain_control_mixed']

EPISODES_PER_SCENARIO = 10
SEED = 7
RERUN_EVAL = True
RUN_ZERO_BASELINE = True

GNBS = [0, 1, 2]
SLICES = ['eMBB', 'URLLC', 'mMTC']
CONTROLLER_COLORS = {'Zero action': '#4c78a8', 'Trained PPO': '#f58518'}

assert MODEL_PATH.exists(), MODEL_PATH
config = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Repo:', Path.cwd())
print('Run dir:', RUN_DIR)
print('Model:', MODEL_PATH)
print('Output:', OUT_DIR)
print('Scenarios:', SCENARIOS)
display(pd.Series(config, name='value').to_frame().head(80))

In [ ]:
def apply_current_run_env_settings(env_args, config):
    """Make evaluation env match the training run closely."""
    env_args.radio_substeps = int(config.get('radio_substeps', env_args.radio_substeps))
    env_args.radio_tick_seconds = float(config.get('radio_tick_seconds', env_args.radio_tick_seconds))
    env_args.local_steps_per_global = int(config.get('local_steps_per_global', env_args.local_steps_per_global))
    env_args.global_steps_per_episode = int(config.get('global_steps_per_episode', env_args.global_steps_per_episode))
    env_args.upper_window_seconds = float(config.get('upper_window_seconds', env_args.upper_window_seconds))
    env_args.max_handovers_per_local_step = int(config.get('max_handovers_per_local_step', env_args.max_handovers_per_local_step))
    env_args.max_handovers_per_ue_episode = int(config.get('max_handovers_per_ue_episode', env_args.max_handovers_per_ue_episode))
    env_args.max_handovers_per_episode = int(config.get('max_handovers_per_episode', env_args.max_handovers_per_episode))
    env_args.handover_pingpong_guard_s = float(config.get('handover_pingpong_guard_s', env_args.handover_pingpong_guard_s))
    env_args.center_gap_topology = str(config.get('center_gap_topology', env_args.center_gap_topology))
    env_args.safe_admission = bool(config.get('safe_admission', env_args.safe_admission))
    env_args.warmup_steps = int(config.get('warmup_steps', env_args.warmup_steps))
    env_args.post_handover_settle_steps = int(config.get('post_handover_settle_steps', env_args.post_handover_settle_steps))
    env_args.demand_calibration_alpha = float(config.get('demand_calibration_alpha', env_args.demand_calibration_alpha))
    env_args.dynamic_upper_window = bool(config.get('dynamic_upper_window', False))
    env_args.max_dynamic_local_steps_per_global = int(
        config.get('max_dynamic_local_steps_per_global', getattr(env_args, 'max_dynamic_local_steps_per_global', 12))
    )

    # Reward/logging settings. These do not change the deterministic action, but keep reported rewards comparable.
    env_args.upper_reward_mode = str(config.get('upper_reward_mode', env_args.upper_reward_mode))
    env_args.load_balance_reward_weight = float(config.get('load_balance_reward_weight', env_args.load_balance_reward_weight))
    env_args.paper_handover_penalty_weight = float(config.get('paper_handover_penalty_weight', env_args.paper_handover_penalty_weight))
    env_args.paper_pingpong_penalty_weight = float(config.get('paper_pingpong_penalty_weight', env_args.paper_pingpong_penalty_weight))
    env_args.paper_excess_load_penalty_weight = float(config.get('paper_excess_load_penalty_weight', env_args.paper_excess_load_penalty_weight))
    env_args.contradictory_bias_penalty_weight = float(config.get('contradictory_bias_penalty_weight', env_args.contradictory_bias_penalty_weight))
    env_args.idle_slice_bias_penalty_weight = float(config.get('idle_slice_bias_penalty_weight', env_args.idle_slice_bias_penalty_weight))
    env_args.sinr_penalty_weight = float(config.get('sinr_penalty_weight', getattr(env_args, 'sinr_penalty_weight', 0.0)))
    env_args.sinr_floor_db = float(config.get('sinr_floor_db', getattr(env_args, 'sinr_floor_db', 5.0)))
    env_args.expert_bias_reward_weight = float(config.get('expert_bias_reward_weight', getattr(env_args, 'expert_bias_reward_weight', 0.0)))
    env_args.expert_bias_closeness_threshold = float(config.get('expert_bias_closeness_threshold', getattr(env_args, 'expert_bias_closeness_threshold', 0.95)))
    env_args.expert_bias_csv = Path(config.get('expert_bias_csv', getattr(env_args, 'expert_bias_csv', ''))) if config.get('expert_bias_csv', None) else None
    return env_args


def make_zero_action_policy(scenario_name):
    env_args = apply_current_run_env_settings(build_args(scenario_name, SEED), config)
    env = make_env(env_args)
    try:
        return ZeroActionPolicy(int(np.prod(env.action_space.shape)))
    finally:
        env.close()


def run_controller_eval(label, model_or_policy, scenario_name):
    env_args = apply_current_run_env_settings(build_args(scenario_name, SEED), config)
    env = make_env(env_args)
    trace_path = OUT_DIR / f"{scenario_name}_{label.lower().replace(' ', '_')}_trace.csv"
    try:
        stats = evaluate_upper_policy(
            model_or_policy,
            env,
            n_eval_episodes=EPISODES_PER_SCENARIO,
            validation_csv=trace_path,
        )
    finally:
        env.close()
    stats['scenario_name'] = scenario_name
    stats['controller'] = label
    stats['trace_csv'] = str(trace_path)
    return stats

if RERUN_EVAL:
    model = PPO.load(str(MODEL_PATH), device='cpu')
    summary_rows = []
    for scenario_name in SCENARIOS:
        print(f'Evaluating trained PPO on {scenario_name}')
        summary_rows.append(run_controller_eval('Trained PPO', model, scenario_name))

        if RUN_ZERO_BASELINE:
            print(f'Evaluating zero action on {scenario_name}')
            summary_rows.append(run_controller_eval('Zero action', make_zero_action_policy(scenario_name), scenario_name))

    eval_summary = pd.DataFrame(summary_rows)
    summary_path = OUT_DIR / 'evaluation_summary.csv'
    eval_summary.to_csv(summary_path, index=False)
    print('Saved:', summary_path)
else:
    eval_summary = pd.read_csv(OUT_DIR / 'evaluation_summary.csv')

display(eval_summary)

In [ ]:
trace_files = sorted(OUT_DIR.glob('*_trace.csv'))
assert trace_files, f'No trace CSV files found in {OUT_DIR}'

frames = []
for path in trace_files:
    df = pd.read_csv(path)
    name = path.name
    controller = 'Trained PPO' if '_trained_ppo_' in name else 'Zero action'
    scenario = name.replace('_trained_ppo_trace.csv', '').replace('_zero_action_trace.csv', '')
    if 'scenario_name' not in df.columns:
        df['scenario_name'] = scenario
    df['controller'] = controller
    df['trace_file'] = str(path)
    frames.append(df)

trace = pd.concat(frames, ignore_index=True)
combined_path = OUT_DIR / 'combined_trace.csv'
trace.to_csv(combined_path, index=False)

print(trace.shape)
print('Saved:', combined_path)
display(trace[['controller', 'scenario_name', 'episode', 'step', 'reward', 'handover_count', 'network_demand_prb_start', 'network_demand_prb_end', 'network_delivery_ratio']].head(20))

In [ ]:
summary_cols = [
    'reward',
    'paper_cost_reward',
    'expert_bias_reward',
    'expert_bias_closeness',
    'handover_count',
    'paper_demand_load_std',
    'paper_useful_load_std',
    'paper_excess_load_mean',
    'global_cost_improvement',
    'network_demand_prb_start',
    'network_demand_prb_end',
    'network_used_prb_start',
    'network_used_prb_end',
    'network_throughput_mbps',
    'network_offered_mbps',
    'network_delivery_ratio',
    'network_completed_delay_ms',
    'network_mean_hol_delay_ms',
    'network_max_hol_delay_ms',
    'network_queue_kbits',
    'network_drop_ratio',
    'network_packet_failure_ratio',
]
summary_cols = [c for c in summary_cols if c in trace.columns]

network_summary = (
    trace.groupby(['controller', 'scenario_name'])[summary_cols]
    .agg(['mean', 'std', 'min', 'max'])
    .reset_index()
)
network_summary.columns = [
    '_'.join(part for part in col if part).rstrip('_')
    if isinstance(col, tuple) else col
    for col in network_summary.columns
]
network_summary_path = OUT_DIR / 'network_qos_reward_summary.csv'
network_summary.to_csv(network_summary_path, index=False)

print('Saved:', network_summary_path)
display(network_summary.round(4))

In [ ]:
def build_before_after_long(kind):
    rows = []
    for _, row in trace.iterrows():
        for phase in ['start', 'end']:
            for gnb in GNBS:
                for slice_type in SLICES:
                    rows.append({
                        'controller': row['controller'],
                        'scenario_name': row['scenario_name'],
                        'episode': row.get('episode', np.nan),
                        'step': row.get('step', np.nan),
                        'phase': phase,
                        'gnb_id': gnb,
                        'slice_type': slice_type,
                        f'{kind}_prb': row.get(f'{kind}_prb_{phase}_g{gnb}_{slice_type}', np.nan),
                        f'gnb_total_{kind}_prb': row.get(f'gnb_{kind}_prb_{phase}_g{gnb}', np.nan),
                        'handover_count': row.get('handover_count', np.nan),
                        'reward': row.get('reward', np.nan),
                        'ue_count': row.get(f'ue_count_g{gnb}_{slice_type}', np.nan),
                    })
    out = pd.DataFrame(rows)
    return out.dropna(subset=[f'{kind}_prb'], how='all')

demand_long = build_before_after_long('demand')
used_long = build_before_after_long('used')

demand_summary = (
    demand_long.groupby(['controller', 'scenario_name', 'phase', 'gnb_id', 'slice_type'], as_index=False)
    .agg(demand_prb_mean=('demand_prb', 'mean'), demand_prb_std=('demand_prb', 'std'), ue_count_mean=('ue_count', 'mean'))
)
used_summary = (
    used_long.groupby(['controller', 'scenario_name', 'phase', 'gnb_id', 'slice_type'], as_index=False)
    .agg(used_prb_mean=('used_prb', 'mean'), used_prb_std=('used_prb', 'std'), ue_count_mean=('ue_count', 'mean'))
)

demand_long.to_csv(OUT_DIR / 'demand_before_after_long.csv', index=False)
demand_summary.to_csv(OUT_DIR / 'demand_before_after_summary.csv', index=False)
used_long.to_csv(OUT_DIR / 'used_prb_before_after_long.csv', index=False)
used_summary.to_csv(OUT_DIR / 'used_prb_before_after_summary.csv', index=False)

print('Saved demand/used before-after CSVs')
display(demand_summary.head(30).round(3))

In [ ]:
qos_rows = []
for _, row in trace.iterrows():
    for gnb in GNBS:
        for slice_type in SLICES:
            qos_rows.append({
                'controller': row['controller'],
                'scenario_name': row['scenario_name'],
                'episode': row.get('episode', np.nan),
                'step': row.get('step', np.nan),
                'gnb_id': gnb,
                'slice_type': slice_type,
                'throughput_mbps': row.get(f'qos_throughput_mbps_g{gnb}_{slice_type}', np.nan),
                'offered_mbps': row.get(f'qos_offered_mbps_g{gnb}_{slice_type}', np.nan),
                'delivery_ratio': row.get(f'qos_delivery_ratio_g{gnb}_{slice_type}', np.nan),
                'completed_delay_ms': row.get(f'qos_completed_delay_ms_g{gnb}_{slice_type}', np.nan),
                'mean_hol_delay_ms': row.get(f'qos_mean_hol_delay_ms_g{gnb}_{slice_type}', np.nan),
                'max_hol_delay_ms': row.get(f'qos_max_hol_delay_ms_g{gnb}_{slice_type}', np.nan),
                'queue_kbits': row.get(f'qos_queue_kbits_g{gnb}_{slice_type}', np.nan),
                'drop_ratio': row.get(f'qos_drop_ratio_g{gnb}_{slice_type}', np.nan),
                'packet_failure_ratio': row.get(f'qos_packet_failure_ratio_g{gnb}_{slice_type}', np.nan),
                'sinr_db': row.get(f'qos_sinr_db_g{gnb}_{slice_type}', np.nan),
                'rsrq_db': row.get(f'qos_rsrq_db_g{gnb}_{slice_type}', np.nan),
                'demand_prb_start': row.get(f'demand_prb_start_g{gnb}_{slice_type}', np.nan),
                'demand_prb_end': row.get(f'demand_prb_end_g{gnb}_{slice_type}', np.nan),
                'used_prb_start': row.get(f'used_prb_start_g{gnb}_{slice_type}', np.nan),
                'used_prb_end': row.get(f'used_prb_end_g{gnb}_{slice_type}', np.nan),
                'ue_count': row.get(f'ue_count_g{gnb}_{slice_type}', np.nan),
            })

qos_long = pd.DataFrame(qos_rows)
qos_active = qos_long[(qos_long['ue_count'].fillna(0) > 0) | (qos_long['demand_prb_end'].fillna(0) > 0)].copy()
qos_summary = (
    qos_active.groupby(['controller', 'scenario_name', 'gnb_id', 'slice_type'], as_index=False)
    .agg({
        'throughput_mbps': 'mean',
        'offered_mbps': 'mean',
        'delivery_ratio': 'mean',
        'completed_delay_ms': 'mean',
        'mean_hol_delay_ms': 'mean',
        'max_hol_delay_ms': 'mean',
        'queue_kbits': 'mean',
        'drop_ratio': 'mean',
        'packet_failure_ratio': 'mean',
        'sinr_db': 'mean',
        'rsrq_db': 'mean',
        'demand_prb_start': 'mean',
        'demand_prb_end': 'mean',
        'used_prb_start': 'mean',
        'used_prb_end': 'mean',
        'ue_count': 'mean',
    })
)

qos_long.to_csv(OUT_DIR / 'qos_per_gnb_slice_long.csv', index=False)
qos_summary.to_csv(OUT_DIR / 'qos_per_gnb_slice_summary.csv', index=False)
print('Saved QoS CSVs')
display(qos_summary.round(3))

In [ ]:
slice_qos_rows = []
for _, row in trace.iterrows():
    for slice_type in SLICES:
        slice_qos_rows.append({
            'controller': row['controller'],
            'scenario_name': row['scenario_name'],
            'episode': row.get('episode', np.nan),
            'step': row.get('step', np.nan),
            'slice_type': slice_type,
            'demand_prb_start': row.get(f'slice_demand_prb_start_{slice_type}', np.nan),
            'demand_prb_end': row.get(f'slice_demand_prb_end_{slice_type}', np.nan),
            'used_prb_start': row.get(f'slice_used_prb_start_{slice_type}', np.nan),
            'used_prb_end': row.get(f'slice_used_prb_end_{slice_type}', np.nan),
            'throughput_mbps': row.get(f'qos_slice_throughput_mbps_{slice_type}', np.nan),
            'offered_mbps': row.get(f'qos_slice_offered_mbps_{slice_type}', np.nan),
            'delivery_ratio': row.get(f'qos_slice_delivery_ratio_{slice_type}', np.nan),
            'completed_delay_ms': row.get(f'qos_slice_completed_delay_ms_{slice_type}', np.nan),
            'mean_hol_delay_ms': row.get(f'qos_slice_mean_hol_delay_ms_{slice_type}', np.nan),
            'max_hol_delay_ms': row.get(f'qos_slice_max_hol_delay_ms_{slice_type}', np.nan),
            'queue_kbits': row.get(f'qos_slice_queue_kbits_{slice_type}', np.nan),
            'drop_ratio': row.get(f'qos_slice_drop_ratio_{slice_type}', np.nan),
            'packet_failure_ratio': row.get(f'qos_slice_packet_failure_ratio_{slice_type}', np.nan),
            'sinr_db': row.get(f'qos_slice_sinr_db_{slice_type}', np.nan),
            'rsrq_db': row.get(f'qos_slice_rsrq_db_{slice_type}', np.nan),
        })

slice_qos = pd.DataFrame(slice_qos_rows)
slice_qos_summary = slice_qos.groupby(['controller', 'scenario_name', 'slice_type'], as_index=False).mean(numeric_only=True)
slice_qos.to_csv(OUT_DIR / 'qos_per_slice_long.csv', index=False)
slice_qos_summary.to_csv(OUT_DIR / 'qos_per_slice_summary.csv', index=False)
display(slice_qos_summary.round(3))

In [ ]:
# Network QoS: zero-action baseline vs trained PPO.
plot_df = trace.groupby(['controller', 'scenario_name'], as_index=False).mean(numeric_only=True)
metrics = [
    ('network_throughput_mbps', 'Network throughput (Mbps)', None),
    ('network_delivery_ratio', 'Network delivery ratio', (0, 1.05)),
    ('network_mean_hol_delay_ms', 'Network mean HOL delay (ms)', None),
    ('network_queue_kbits', 'Network queue (kbits)', None),
    ('handover_count', 'Handovers per upper window', None),
    ('paper_demand_load_std', 'Demand-load std after action', None),
]
metrics = [(c, t, y) for c, t, y in metrics if c in plot_df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 8), constrained_layout=True)
axes = axes.ravel()
x = np.arange(len(SCENARIOS))
width = 0.36
for ax, (metric, title, ylim) in zip(axes, metrics):
    for idx, controller in enumerate(['Zero action', 'Trained PPO']):
        vals = []
        for scenario in SCENARIOS:
            sub = plot_df[(plot_df['controller'] == controller) & (plot_df['scenario_name'] == scenario)]
            vals.append(float(sub[metric].iloc[0]) if len(sub) else np.nan)
        ax.bar(x + (idx - 0.5) * width, vals, width, label=controller, color=CONTROLLER_COLORS[controller])
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(SCENARIOS, rotation=35, ha='right')
    ax.grid(axis='y', alpha=0.25)
    if ylim is not None:
        ax.set_ylim(*ylim)
for ax in axes[len(metrics):]:
    ax.axis('off')
axes[0].legend()
network_plot_path = OUT_DIR / 'network_qos_zero_vs_trained.png'
plt.savefig(network_plot_path, dpi=160, bbox_inches='tight')
print('Saved:', network_plot_path)
plt.show()

In [ ]:
# Demand and used PRB before/after for the trained model.
for scenario_name in SCENARIOS:
    trained_demand = demand_summary[(demand_summary['controller'] == 'Trained PPO') & (demand_summary['scenario_name'] == scenario_name)].copy()
    trained_used = used_summary[(used_summary['controller'] == 'Trained PPO') & (used_summary['scenario_name'] == scenario_name)].copy()

    fig, axes = plt.subplots(len(SLICES), 2, figsize=(15, 3.2 * len(SLICES)), constrained_layout=True)
    for r, slice_type in enumerate(SLICES):
        for c, (kind, frame, value_col, title_prefix) in enumerate([
            ('demand', trained_demand, 'demand_prb_mean', 'Demand PRB'),
            ('used', trained_used, 'used_prb_mean', 'Used PRB'),
        ]):
            ax = axes[r, c]
            sub = frame[frame['slice_type'] == slice_type]
            starts = [sub[(sub['phase'] == 'start') & (sub['gnb_id'] == g)][value_col].mean() for g in GNBS]
            ends = [sub[(sub['phase'] == 'end') & (sub['gnb_id'] == g)][value_col].mean() for g in GNBS]
            g = np.arange(len(GNBS))
            ax.bar(g - 0.18, starts, 0.36, label='start', color='#72b7b2')
            ax.bar(g + 0.18, ends, 0.36, label='end', color='#e45756')
            ax.set_title(f'{scenario_name} | {slice_type} | {title_prefix}')
            ax.set_xticks(g)
            ax.set_xticklabels([f'g{item}' for item in GNBS])
            ax.grid(axis='y', alpha=0.25)
            if r == 0 and c == 0:
                ax.legend()
    path = OUT_DIR / f'{scenario_name}_trained_demand_used_before_after.png'
    plt.savefig(path, dpi=160, bbox_inches='tight')
    print('Saved:', path)
    plt.show()

In [ ]:
# Per-slice QoS: zero action vs trained PPO.
qos_metrics = [
    ('throughput_mbps', 'Throughput (Mbps)', None),
    ('delivery_ratio', 'Delivery ratio', (0, 1.05)),
    ('mean_hol_delay_ms', 'Mean HOL delay (ms)', None),
    ('queue_kbits', 'Queue (kbits)', None),
    ('sinr_db', 'SINR (dB)', None),
    ('rsrq_db', 'RSRQ (dB)', None),
]

for scenario_name in SCENARIOS:
    sub = slice_qos_summary[slice_qos_summary['scenario_name'] == scenario_name]
    fig, axes = plt.subplots(2, 3, figsize=(18, 8), constrained_layout=True)
    axes = axes.ravel()
    x = np.arange(len(SLICES))
    width = 0.36
    for ax, (metric, title, ylim) in zip(axes, qos_metrics):
        for idx, controller in enumerate(['Zero action', 'Trained PPO']):
            vals = []
            for slice_type in SLICES:
                row = sub[(sub['controller'] == controller) & (sub['slice_type'] == slice_type)]
                vals.append(float(row[metric].iloc[0]) if len(row) and metric in row else np.nan)
            ax.bar(x + (idx - 0.5) * width, vals, width, label=controller, color=CONTROLLER_COLORS[controller])
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(SLICES)
        ax.grid(axis='y', alpha=0.25)
        if ylim is not None:
            ax.set_ylim(*ylim)
    axes[0].legend()
    path = OUT_DIR / f'{scenario_name}_per_slice_qos_zero_vs_trained.png'
    plt.savefig(path, dpi=160, bbox_inches='tight')
    print('Saved:', path)
    plt.show()

In [ ]:
# Per-gNB/per-slice QoS heatmaps for trained PPO.
heatmap_metrics = ['throughput_mbps', 'delivery_ratio', 'mean_hol_delay_ms', 'queue_kbits', 'sinr_db', 'rsrq_db']
for scenario_name in SCENARIOS:
    sub = qos_summary[(qos_summary['controller'] == 'Trained PPO') & (qos_summary['scenario_name'] == scenario_name)]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
    axes = axes.ravel()
    for ax, metric in zip(axes, heatmap_metrics):
        matrix = np.full((len(GNBS), len(SLICES)), np.nan)
        for i, gnb in enumerate(GNBS):
            for j, slice_type in enumerate(SLICES):
                row = sub[(sub['gnb_id'] == gnb) & (sub['slice_type'] == slice_type)]
                if len(row):
                    matrix[i, j] = float(row[metric].iloc[0])
        im = ax.imshow(matrix, aspect='auto')
        ax.set_title(metric)
        ax.set_xticks(np.arange(len(SLICES)))
        ax.set_xticklabels(SLICES)
        ax.set_yticks(np.arange(len(GNBS)))
        ax.set_yticklabels([f'g{g}' for g in GNBS])
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                val = matrix[i, j]
                label = '' if np.isnan(val) else f'{val:.2f}'
                ax.text(j, i, label, ha='center', va='center', color='white' if np.isfinite(val) and val < np.nanmean(matrix) else 'black')
        fig.colorbar(im, ax=ax, shrink=0.75)
    path = OUT_DIR / f'{scenario_name}_trained_qos_heatmaps.png'
    plt.savefig(path, dpi=160, bbox_inches='tight')
    print('Saved:', path)
    plt.show()

In [ ]:
# Learned directional biases from the trained model trace.
bias_cols = [c for c in trace.columns if c.startswith('bias_')]
trained_bias = trace[trace['controller'] == 'Trained PPO'].copy()
if bias_cols:
    bias_summary = (
        trained_bias.groupby('scenario_name')[bias_cols]
        .agg(['mean', 'std', 'min', 'max'])
    )
    bias_summary.columns = ['_'.join(col).rstrip('_') for col in bias_summary.columns]
    bias_summary_path = OUT_DIR / 'trained_directional_bias_summary.csv'
    bias_summary.reset_index().to_csv(bias_summary_path, index=False)
    print('Saved:', bias_summary_path)

    for scenario_name in SCENARIOS:
        sub = trained_bias[trained_bias['scenario_name'] == scenario_name]
        means = sub[bias_cols].mean().sort_values()
        fig, ax = plt.subplots(figsize=(13, 6))
        means.plot(kind='bar', ax=ax, color=np.where(means.values < 0, '#e45756', '#72b7b2'))
        ax.axhline(0, color='black', lw=1)
        ax.set_title(f'{scenario_name}: mean learned directional bias')
        ax.set_ylabel('bias [-1, 1]')
        ax.tick_params(axis='x', rotation=70)
        ax.grid(axis='y', alpha=0.25)
        path = OUT_DIR / f'{scenario_name}_trained_directional_bias_mean.png'
        plt.savefig(path, dpi=160, bbox_inches='tight')
        print('Saved:', path)
        plt.show()
else:
    print('No bias_* columns found.')

In [ ]:
print('Artifacts written under:', OUT_DIR)
for path in sorted(OUT_DIR.glob('*')):
    print(path)